<a href="https://colab.research.google.com/github/rizkiismail9a/data-science-2026-unsia/blob/main/pertemuan_6_MuhamadRizkiIsmail_240401010126.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Encoding Kategorikal
# Label Encoding
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.DataFrame({
'Gender': ['male', 'female', 'female', 'male', 'female'],
'Survived': [0, 1, 1, 0, 1]
})

# Label Encoding
le = LabelEncoder()

# fit_transform: belajar pemetaan + langsung transformasi
df['Gender_enc'] = le.fit_transform(df['Gender'])
print(df)
# male akan memiliki kategori 1 karena secara alfabet, female lebih dulu muncul

   Gender  Survived  Gender_enc
0    male         0           1
1  female         1           0
2  female         1           0
3    male         0           1
4  female         1           0


In [ ]:
# One Hot Encoding

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
df = pd.DataFrame({'City': ['Jakarta','Surabaya','Bandung','Jakarta','Surabaya']})

# ── Cara 1: Pandas get_dummies (paling mudah) ─────────────────────
df_ohe = pd.get_dummies(df,
columns=['City'],
drop_first=False, # True untuk hindari dummy variable trap; Itu adalah keadaan ketika ada variable yang tidak dibutuhkan
dtype=int) # hasilkan 0/1 bukan True/False
print(df_ohe)

# Dengn ini City_Bandung akan dibuang
# Jika ingin tahu value City_Bandung, cukup ambil dari value Jakarta dan Surabaya
df_ohe_drop = pd.get_dummies(df, columns=['City'],
drop_first=True, dtype=int)
print(df_ohe_drop)

   City_Jakarta  City_Surabaya
0             1              0
1             0              1
2             0              0
3             1              0
4             0              1


In [5]:
# Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

df = pd.DataFrame({
'Education': ['SMA', 'S1', 'SD', 'D3', 'S2', 'SMP', 'TK'],
'Gaji_Juta': [5, 12, 3, 8, 18, 4, 1]
                    })

# Definisikan urutan kategori secara eksplisit
edu_order = [['SD','SMP','SMA','D3', 'S1', 'S2']]

enc = OrdinalEncoder(
    categories=edu_order,
    handle_unknown='use_encoded_value',
    unknown_value=-1 # Jika ada kategori baru, maka tambahkan -1
)

df['Pendidikan_Enc'] = enc.fit_transform(df[['Education']])
print(df.sort_values('Pendidikan_Enc'))

  Education  Gaji_Juta  Pendidikan_Enc
6        TK          1            -1.0
2        SD          3             0.0
5       SMP          4             1.0
0       SMA          5             2.0
3        D3          8             3.0
1        S1         12             4.0
4        S2         18             5.0


In [6]:
# Scaling dan Normalisasi
# MinMaxScaler

from sklearn.preprocessing import MinMaxScaler
import pandas as pd

df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12] # dalam juta rupiah
})

scaler = MinMaxScaler(feature_range=(0,1)) # Ini default, (0,1) gak ditulis pun gapapa
# fit_transform: belajar min/max + transformasi data
X_scaled = scaler.fit_transform(df[['Usia', 'Pendapatan']])

print('Min per fitur :', scaler.data_min_) # [25 5]
print('Max per fitur :', scaler.data_max_) # [55 35]
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_sc', 'Pendapatan_sc']).round(3))

Min per fitur : [25.  5.]
Max per fitur : [55. 35.]

   Usia_sc  Pendapatan_sc
0    0.000          0.000
1    0.667          0.500
2    0.233          0.100
3    1.000          1.000
4    0.100          0.233


In [7]:
# Z-score strandarization
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12]
})

scaler = StandardScaler()
X = df[['Usia', 'Pendapatan']]

X_scaled = scaler.fit_transform(X)
print('Mean per fitur:', scaler.mean_) # rata-rata setiap kolom
print('Scale per fitur:', scaler.scale_) # std dev setiap kolom
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_z', 'Pend_z']).round(3))

Mean per fitur: [37. 16.]
Scale per fitur: [11.296017   10.75174404]

   Usia_z  Pend_z
0  -1.062  -1.023
1   0.708   0.372
2  -0.443  -0.744
3   1.593   1.767
4  -0.797  -0.372


In [23]:
# Latihan
import pandas as pd, seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = sns.load_dataset('titanic')

# print(df)

# kolom yang dipakai
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()

print('Shape: ', df.shape)
print('\nMissing value: ')
print(df.isnull().sum())
print('\nDistribusi Target: ')
print(df['survived'].value_counts(normalize=True).round(3))

# Handling Missing Values
# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked isi dengan modus
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('\nMissing setelah handling:')
print(df.isnull().sum())


# Terapkan One-Hot Encoding untuk gender dan embarked
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True, dtype=int)
print('\nKolom setelah encoding:')
print(df.columns.tolist())


# Split train-test
X = df.drop('survived', axis=1) # semua kolom selain kolomg survived adalah sumbu X
y = df['survived']

# mirip Object Destructuring javascript
X_train, X_test, y_train, y_test = train_test_split(
X, y,
test_size=0.2,
random_state=42,
stratify=y # proporsi kelas terjaga
)

print(f'\nTrain: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')
print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

# Scaling
# Hanya kolom numerik yang perlu di-scale
# Kolom biner (sex_male, embarked_Q, embarked_S) TIDAK perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']
scaler = StandardScaler()

# fit_transform pada training set (belajar μ dan σ dari sini)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
# transform saja pada test set (gunakan μ dan σ dari training!)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('\nMean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train):', scaler.scale_.round(2))
print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))

print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}, y_test : {y_test.shape}')


Shape:  (891, 8)

Missing value: 
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi Target: 
survived
0    0.616
1    0.384
Name: proportion, dtype: float64

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.46